# 01 - 数据获取与探索

本 Notebook 演示如何：
1. 使用 akshare 获取 A 股历史日K线数据
2. 清洗和预处理数据
3. 初步探索数据特征
4. 将数据存储为 Parquet 格式

In [ ]:
# 设置路径
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

from config.settings import WATCHLIST, DEFAULT_START_DATE, DEFAULT_END_DATE
from src.data.fetcher import fetch_daily_kline
from src.data.cleaner import clean_kline_data, validate_data, add_return_columns
from src.data.storage import save_to_parquet, load_from_parquet

print('✅ 模块加载成功')

## 1. 下载单只股票数据

以贵州茅台 (600519) 为例

In [ ]:
# 下载茅台数据（前复权）
symbol = '600519'
raw_df = fetch_daily_kline(symbol, DEFAULT_START_DATE, DEFAULT_END_DATE, adjust='qfq')
print(f'原始数据形状: {raw_df.shape}')
print(f'\n原始列名: {raw_df.columns.tolist()}')
raw_df.head()

## 2. 清洗数据

In [ ]:
# 清洗数据：中英文列名转换、类型转换、缺失值处理
clean_df = clean_kline_data(raw_df, symbol)
# 添加收益率列
clean_df = add_return_columns(clean_df)
print(f'清洗后数据形状: {clean_df.shape}')
print(f'\n英文列名: {clean_df.columns.tolist()}')
clean_df.head()

In [ ]:
# 验证数据质量
report = validate_data(clean_df, symbol)
for key, value in report.items():
    print(f'{key}: {value}')

## 3. 保存数据

In [ ]:
# 保存为 Parquet 格式（高效读写）
save_to_parquet(clean_df, symbol)

# 验证读取
loaded_df = load_from_parquet(symbol)
print(f'加载数据: {len(loaded_df)} 条')
loaded_df.head()

## 4. 快速数据探索

In [ ]:
# 收盘价走势
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 收盘价
axes[0, 0].plot(clean_df['date'], clean_df['close'], linewidth=0.8)
axes[0, 0].set_title(f'{symbol} 收盘价走势')
axes[0, 0].set_ylabel('价格 (元)')
axes[0, 0].grid(True, alpha=0.3)

# 日收益率分布
axes[0, 1].hist(clean_df['daily_return'].dropna() * 100, bins=100, edgecolor='white')
axes[0, 1].set_title('日收益率分布')
axes[0, 1].set_xlabel('收益率 (%)')
axes[0, 1].grid(True, alpha=0.3)

# 成交量
axes[1, 0].bar(clean_df['date'], clean_df['volume_lots'], width=0.8)
axes[1, 0].set_title('成交量 (手)')
axes[1, 0].grid(True, alpha=0.3)

# 累计收益
axes[1, 1].plot(clean_df['date'], clean_df['cumulative_return'] * 100, linewidth=0.8, color='green')
axes[1, 1].set_title('累计收益率')
axes[1, 1].set_ylabel('收益率 (%)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 5. 批量下载精选股票

下载所有的 WATCHLIST 股票

In [ ]:
from src.data.fetcher import fetch_daily_kline

for symbol, name in WATCHLIST:
    try:
        raw = fetch_daily_kline(symbol, DEFAULT_START_DATE, DEFAULT_END_DATE)
        clean = clean_kline_data(raw, symbol)
        clean = add_return_columns(clean)
        save_to_parquet(clean, symbol)
        print(f'✅ {symbol} {name}: {len(clean)} 条数据')
    except Exception as e:
        print(f'❌ {symbol} {name}: {e}')

In [ ]:
# 对比各股票的累计收益率
fig, ax = plt.subplots(figsize=(12, 6))

for symbol, name in WATCHLIST:
    try:
        df = load_from_parquet(symbol)
        ax.plot(df['date'], df['cumulative_return'] * 100, label=f'{symbol} {name}', linewidth=0.8)
    except FileNotFoundError:
        pass

ax.set_title('精选股票累计收益率对比')
ax.set_xlabel('日期')
ax.set_ylabel('累计收益率 (%)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. 核心发现

> 在此记录你对数据的观察和发现